# Multi-Hop Factual Reasoning: Critical Slowing Down (CSD) Sampling Experiment

**What this demo does:** Analyzes pre-computed LLM responses to multi-hop factual reasoning questions across 6 difficulty levels (1–6 hops) for 3 models (Llama-3.1-8B, GPT-4o-mini, Gemini-2.0-Flash).

The experiment tests whether **Critical Slowing Down** indicators — borrowed from ecology — can detect approaching capability boundaries *before* accuracy drops. CSD indicators include:
- **Variance trace** of semantic embeddings
- **Hartigan's dip test** for multimodality
- **Silhouette score** (k=2 clustering)
- **Bimodality coefficient** and **Ashman's D**
- **Self-consistency disagreement** (baseline)

The pipeline: load pre-computed responses → extract & evaluate answers → compute sentence embeddings → compute CSD indicator battery → analyze cross-level trends → visualize.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('sentence-transformers==3.4.1')
_pip('diptest==0.8.0')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import gc
import json
import math
import os
import random
import re
import time
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kendalltau, skew, kurtosis
from scipy.spatial.distance import pdist
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
import diptest

## Load Experiment Data

Load pre-computed multi-hop reasoning experiment results (18 examples: 3 models × 6 difficulty levels, each with 10 LLM responses).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-276cb0-flickering-before-failing-ecological-cri/main/experiment_iter2_multi_hop_factu/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
examples = data["datasets"][0]["examples"]
metadata = data["metadata"]
print(f"Loaded {len(examples)} examples across {len(metadata['models'])} models and {len(metadata['difficulty_levels'])} difficulty levels")
print(f"Models: {metadata['models']}")
print(f"Difficulty levels (hops): {metadata['difficulty_levels']}")

## Configuration

Tunable parameters for the analysis. These match the original experiment settings.

In [ ]:
# --- Config ---
SEED = 42
DIFFICULTY_LEVELS = [1, 2, 3, 4, 5, 6]
MODELS = [
    "meta-llama/llama-3.1-8b-instruct",
    "openai/gpt-4o-mini",
    "google/gemini-2.0-flash-001",
]
MODEL_SHORT = ["llama-8b", "gpt-4o-mini", "gemini-flash"]
MODEL_TIERS = ["small", "medium", "large"]

# Embedding model for semantic analysis
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
EMBEDDING_BATCH_SIZE = 64  # batch size for sentence-transformers encode

# Number of responses per problem in the pre-computed data
N_RESPONSES_PER_PROBLEM = 10  # original: 10

# KMeans clusters for silhouette score
N_CLUSTERS = 2
KMEANS_N_INIT = 10  # original: 10

# Bimodality consensus threshold
BIMODALITY_DIP_ALPHA = 0.05
BIMODALITY_SIL_THRESHOLD = 0.3
BIMODALITY_BC_THRESHOLD = 5 / 9  # ~0.5556

random.seed(SEED)
np.random.seed(SEED)

## Step 1: Answer Extraction & Fuzzy Matching

Extract final answers from LLM responses using pattern matching (looks for "ANSWER:", common answer phrases, or falls back to last line). Evaluate correctness via multi-criteria fuzzy matching: exact match, substring containment, and token-level F1.

In [ ]:
def extract_answer(response_text: str) -> str:
    """Extract final answer from response text."""
    if not response_text or not response_text.strip():
        return ""

    # Strategy 1: "ANSWER: " prefix (case-insensitive)
    match = re.search(r'ANSWER:\s*(.+?)(?:\n|$)', response_text, re.IGNORECASE)
    if match:
        return match.group(1).strip().rstrip('.')

    # Strategy 2: Common answer patterns
    patterns = [
        r'(?:the\s+)?(?:final\s+)?answer\s+is[:\s]+(.+?)(?:\.|$)',
        r'(?:therefore|thus|so|hence)[,\s]+(?:the\s+answer\s+is\s+)?(.+?)(?:\.|$)',
    ]
    for pat in patterns:
        m = re.search(pat, response_text, re.IGNORECASE)
        if m:
            return m.group(1).strip().rstrip('.')

    # Strategy 3: Last non-empty line (fallback)
    lines = [line.strip() for line in response_text.strip().split('\n') if line.strip()]
    if lines:
        last = lines[-1]
        return last[:100] if len(last) > 100 else last
    return ""


def _clean_answer(text: str) -> str:
    """Normalize answer for comparison."""
    t = text.lower().strip().rstrip('.')
    for prefix in ["the ", "a ", "an ", "in ", "at ", "on "]:
        if t.startswith(prefix):
            t = t[len(prefix):]
    return t.strip()


def fuzzy_match(predicted: str, ground_truth: str, aliases_str: str) -> tuple:
    """Multi-criteria fuzzy matching. Returns (is_correct, f1_score)."""
    pred_clean = _clean_answer(predicted)
    gt_clean = _clean_answer(ground_truth)

    if not pred_clean:
        return False, 0.0

    try:
        aliases = json.loads(aliases_str) if aliases_str else []
    except (json.JSONDecodeError, TypeError):
        aliases = []
    all_answers = [gt_clean] + [_clean_answer(a) for a in aliases]

    for ans in all_answers:
        if pred_clean == ans:
            return True, 1.0
        if ans and pred_clean and (ans in pred_clean or pred_clean in ans):
            return True, 0.9

    pred_tokens = set(pred_clean.split())
    gt_tokens = set(gt_clean.split())
    if pred_tokens and gt_tokens:
        common = pred_tokens & gt_tokens
        precision = len(common) / len(pred_tokens)
        recall = len(common) / len(gt_tokens)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        if f1 >= 0.5:
            return True, f1

    return False, 0.0


# Quick unit test
test_cases = [
    ("ANSWER: Project Mercury", "Project Mercury", "[]", True),
    ("the thames", "Thames", "[]", True),
    ("wrong answer", "Thames", "[]", False),
]
print("Unit tests:")
for pred, gt, aliases, expected in test_cases:
    ans = extract_answer(pred) if "ANSWER:" in pred else pred
    correct, f1 = fuzzy_match(ans, gt, aliases)
    status = "PASS" if correct == expected else "FAIL"
    print(f"  {status}: '{ans}' vs '{gt}' -> correct={correct} (expected={expected})")

## Step 2: Evaluate Pre-computed Responses

Parse the pre-computed responses from each example, extract answers, and evaluate correctness. Group results by (model, difficulty level).

In [ ]:
# Parse examples and group by (model, level)
# Each example has pre-computed responses as JSON strings
results_by_model_level = defaultdict(list)

for ex in examples:
    model = ex["metadata_model"]
    level = ex["metadata_difficulty_level"]
    responses = json.loads(ex["predict_responses"])
    ground_truth = ex["output"]
    aliases = ex.get("metadata_answer_aliases", "[]")

    # Re-extract answers and evaluate (demonstrating the pipeline)
    extracted = [extract_answer(r) for r in responses]
    correctness = []
    f1_scores = []
    for ans in extracted:
        is_correct, f1 = fuzzy_match(ans, ground_truth, aliases)
        correctness.append(is_correct)
        f1_scores.append(f1)

    results_by_model_level[(model, level)].append({
        "question": ex["input"],
        "ground_truth": ground_truth,
        "responses": responses,
        "extracted_answers": extracted,
        "correctness": correctness,
        "f1_scores": f1_scores,
    })

# Print accuracy summary
print(f"{'Model':<45} {'Level':>5} {'Accuracy':>8} {'Correct':>7}")
print("-" * 70)
for model in MODELS:
    for level in DIFFICULTY_LEVELS:
        probs = results_by_model_level.get((model, level), [])
        all_correct = [c for p in probs for c in p["correctness"]]
        if all_correct:
            acc = sum(all_correct) / len(all_correct)
            print(f"{model:<45} {level:>5} {acc:>8.2%} {sum(all_correct):>4}/{len(all_correct)}")

## Step 3: Compute Semantic Embeddings

Embed all LLM responses using `all-MiniLM-L6-v2` sentence transformer. These embeddings capture semantic similarity between responses and form the basis for CSD indicator computation.

In [ ]:
from sentence_transformers import SentenceTransformer

print(f"Loading sentence-transformers model ({EMBEDDING_MODEL_NAME})...")
t0 = time.time()
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"Model loaded in {time.time() - t0:.1f}s")

# Compute embeddings for each (model, level) group
embeddings_by_model_level = {}
for (model, level), probs in results_by_model_level.items():
    texts = []
    for prob in probs:
        texts.extend(prob["responses"])

    non_empty_indices = [i for i, t in enumerate(texts) if t.strip()]
    non_empty_texts = [texts[i] for i in non_empty_indices]

    if not non_empty_texts:
        embeddings_by_model_level[(model, level)] = np.zeros((len(texts), 384))
        continue

    embs = embed_model.encode(non_empty_texts, batch_size=EMBEDDING_BATCH_SIZE, show_progress_bar=False)

    full_embs = np.zeros((len(texts), embs.shape[1]))
    for idx, emb_idx in enumerate(non_empty_indices):
        full_embs[emb_idx] = embs[idx]

    embeddings_by_model_level[(model, level)] = full_embs

print(f"Computed embeddings for {len(embeddings_by_model_level)} (model, level) groups")

# Free model memory
del embed_model
gc.collect()
print("Embedding model freed.")

## Step 4: Compute CSD Indicator Battery

For each (model, level), compute the full battery of Critical Slowing Down indicators:
- **Variance trace**: total variance in embedding space (covariance trace)
- **Hartigan's dip test**: tests for multimodality in the PC1 projection
- **Silhouette score**: quality of k=2 clustering
- **Bimodality coefficient**: skewness/kurtosis-based bimodality measure
- **Ashman's D**: separation of 2-component Gaussian mixture
- **Self-consistency disagreement**: fraction of responses disagreeing with the majority

In [ ]:
def compute_csd_indicators(embeddings, correctness, answers):
    """Compute full CSD indicator battery for one (model, level) group."""
    N = embeddings.shape[0]

    # Filter out zero-vector rows (from empty responses)
    norms = np.linalg.norm(embeddings, axis=1)
    valid_mask = norms > 1e-8
    valid_embeddings = embeddings[valid_mask]
    valid_correctness = [c for c, v in zip(correctness, valid_mask) if v]
    valid_answers = [a for a, v in zip(answers, valid_mask) if v]
    N_valid = valid_embeddings.shape[0]

    if N_valid < 4:
        accuracy = sum(correctness) / len(correctness) if correctness else 0.0
        return {"accuracy": round(accuracy, 4), "n_valid": N_valid,
                "embedding_variance_trace": 0.0, "mean_cosine_distance": 0.0,
                "hartigan_dip_stat": 0.0, "hartigan_dip_pval": 1.0,
                "silhouette_score_k2": -1.0, "bimodality_coefficient": 0.0,
                "ashman_d": 0.0, "bimodality_consensus": False,
                "self_consistency_disagreement": 1.0, "answer_entropy": 0.0}

    # A. Embedding variance
    cov = np.cov(valid_embeddings.T)
    variance_trace = float(np.trace(cov))
    cos_dists = pdist(valid_embeddings, metric='cosine')
    mean_cosine_dist = float(np.mean(cos_dists))

    # B. PC1 projection
    n_components = min(5, N_valid - 1)
    pca = PCA(n_components=max(1, n_components))
    pc_scores = pca.fit_transform(valid_embeddings)
    pc1 = pc_scores[:, 0]

    # C. Hartigan's dip test
    try:
        dip_stat, dip_pval = diptest.diptest(pc1)
    except Exception:
        dip_stat, dip_pval = 0.0, 1.0

    # D. Silhouette score (k=2)
    try:
        kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=SEED, n_init=KMEANS_N_INIT)
        labels = kmeans.fit_predict(valid_embeddings)
        sil_score = float(silhouette_score(valid_embeddings, labels)) if len(set(labels)) == 2 else -1.0
    except Exception:
        sil_score = -1.0

    # E. Bimodality coefficient
    s = float(skew(pc1))
    k = float(kurtosis(pc1))
    n = len(pc1)
    bc_num = s**2 + 1
    bc_den = k + 3 * (n - 1)**2 / ((n - 2) * (n - 3)) if n > 3 else k + 3
    bimodality_coeff = bc_num / bc_den if bc_den != 0 else 0

    # F. Ashman's D (2-component GMM)
    try:
        gmm = GaussianMixture(n_components=2, random_state=SEED)
        gmm.fit(pc1.reshape(-1, 1))
        mu1, mu2 = gmm.means_.flatten()
        s1, s2 = np.sqrt(gmm.covariances_.flatten())
        ashman_d = float(np.sqrt(2) * abs(mu1 - mu2) / np.sqrt(s1**2 + s2**2)) if (s1**2 + s2**2) > 0 else 0
    except Exception:
        ashman_d = 0.0

    # G. Self-consistency disagreement
    if valid_answers:
        answer_counts = Counter(a.lower().strip() for a in valid_answers if a.strip())
        if answer_counts:
            most_common_count = answer_counts.most_common(1)[0][1]
            disagreement_rate = 1.0 - (most_common_count / len(valid_answers))
        else:
            disagreement_rate = 1.0
    else:
        disagreement_rate = 1.0

    # H. Accuracy & entropy
    accuracy = sum(valid_correctness) / len(valid_correctness) if valid_correctness else 0.0
    if valid_answers:
        ac = Counter(a.lower().strip() for a in valid_answers if a.strip())
        total_a = sum(ac.values())
        answer_entropy = -sum((c/total_a) * np.log2(c/total_a) for c in ac.values()) if total_a > 0 else 0.0
    else:
        answer_entropy = 0.0

    # I. Bimodality consensus
    bimodality_consensus = sum([
        dip_pval < BIMODALITY_DIP_ALPHA,
        sil_score > BIMODALITY_SIL_THRESHOLD,
        bimodality_coeff > BIMODALITY_BC_THRESHOLD,
    ]) >= 2

    return {
        "accuracy": round(accuracy, 4), "n_valid": N_valid,
        "embedding_variance_trace": round(variance_trace, 6),
        "mean_cosine_distance": round(mean_cosine_dist, 6),
        "hartigan_dip_stat": round(float(dip_stat), 6),
        "hartigan_dip_pval": round(float(dip_pval), 6),
        "silhouette_score_k2": round(float(sil_score), 4),
        "bimodality_coefficient": round(float(bimodality_coeff), 4),
        "ashman_d": round(float(ashman_d), 4),
        "bimodality_consensus": bimodality_consensus,
        "self_consistency_disagreement": round(float(disagreement_rate), 4),
        "answer_entropy": round(float(answer_entropy), 4),
    }


# Compute indicators for each (model, level) group
indicators_by_model = defaultdict(dict)

for (model, level), probs in results_by_model_level.items():
    embs = embeddings_by_model_level[(model, level)]
    all_correct = [c for p in probs for c in p["correctness"]]
    all_answers = [a for p in probs for a in p["extracted_answers"]]
    indicators_by_model[model][level] = compute_csd_indicators(embs, all_correct, all_answers)

# Print summary table
print(f"{'Model':<20} {'Lvl':>3} {'Acc':>6} {'VarTr':>8} {'DipP':>8} {'Sil':>6} {'BC':>6} {'Disagr':>6}")
print("-" * 75)
for model, short in zip(MODELS, MODEL_SHORT):
    for level in DIFFICULTY_LEVELS:
        ind = indicators_by_model[model].get(level, {})
        if ind:
            print(f"{short:<20} {level:>3} {ind['accuracy']:>6.2f} "
                  f"{ind['embedding_variance_trace']:>8.4f} {ind['hartigan_dip_pval']:>8.4f} "
                  f"{ind['silhouette_score_k2']:>6.3f} {ind['bimodality_coefficient']:>6.3f} "
                  f"{ind['self_consistency_disagreement']:>6.3f}")

## Step 5: Cross-Level Trend Analysis

Compute Kendall tau rank correlations between difficulty level and each CSD indicator. A significant positive trend in variance/disagreement with increasing difficulty would support the CSD hypothesis. We also use the **full experiment** indicators from metadata for the complete N=50 picture.

In [ ]:
def compute_cross_level_trends(level_indicators):
    """Compute Kendall tau trends across difficulty levels."""
    levels = sorted(level_indicators.keys())
    indicator_names = [
        "embedding_variance_trace", "mean_cosine_distance",
        "hartigan_dip_stat", "silhouette_score_k2",
        "bimodality_coefficient", "self_consistency_disagreement",
        "accuracy", "answer_entropy", "ashman_d",
    ]
    results = {}
    for name in indicator_names:
        values = [level_indicators[lv].get(name, 0) for lv in levels]
        try:
            tau, pval = kendalltau(levels, values)
            results[name] = {"tau": round(float(tau), 4), "pval": round(float(pval), 4)}
        except Exception:
            results[name] = {"tau": 0.0, "pval": 1.0}
    return results


# Use the FULL experiment indicators from metadata (N=50 per level) for trend analysis
# These are more statistically meaningful than our demo subset (N=10 per level)
print("=== Kendall Tau Trends (from full experiment, N=50 per level) ===\n")
per_model = metadata["per_model_summary"]

for model, short in zip(MODELS, MODEL_SHORT):
    if model not in per_model:
        continue
    full_indicators = {int(k): v for k, v in per_model[model]["per_level_indicators"].items()}
    trends = compute_cross_level_trends(full_indicators)

    print(f"--- {short} ---")
    print(f"  {'Indicator':<35} {'tau':>7} {'p-value':>8}")
    for name, vals in trends.items():
        sig = " *" if vals["pval"] < 0.05 else ""
        print(f"  {name:<35} {vals['tau']:>7.3f} {vals['pval']:>8.4f}{sig}")
    print()

## Visualization: CSD Indicators Across Difficulty Levels

Plot key CSD indicators alongside accuracy for each model. The hypothesis: CSD indicators (variance, disagreement, bimodality) should increase as models approach their capability boundary — *before* accuracy collapses.

In [ ]:
# Use FULL experiment indicators from metadata for clearer visualization (N=50 per level)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("CSD Indicators vs Difficulty Level (Full Experiment, N=50/level)", fontsize=14, fontweight='bold')

indicators_to_plot = [
    ("accuracy", "Accuracy", False),
    ("embedding_variance_trace", "Embedding Variance Trace", True),
    ("self_consistency_disagreement", "Self-Consistency Disagreement", True),
    ("bimodality_coefficient", "Bimodality Coefficient", True),
    ("silhouette_score_k2", "Silhouette Score (k=2)", True),
    ("answer_entropy", "Answer Entropy", True),
]

colors = ["#e74c3c", "#3498db", "#2ecc71"]
markers = ["o", "s", "^"]

for ax_idx, (ind_name, ind_label, show_threshold) in enumerate(indicators_to_plot):
    ax = axes[ax_idx // 3][ax_idx % 3]

    for m_idx, (model, short) in enumerate(zip(MODELS, MODEL_SHORT)):
        if model not in per_model:
            continue
        level_ind = per_model[model]["per_level_indicators"]
        levels = sorted(int(k) for k in level_ind.keys())
        values = [level_ind[str(lv)][ind_name] for lv in levels]
        ax.plot(levels, values, color=colors[m_idx], marker=markers[m_idx],
                label=short, linewidth=2, markersize=7)

    ax.set_xlabel("Difficulty Level (hops)")
    ax.set_ylabel(ind_label)
    ax.set_title(ind_label)
    ax.set_xticks(DIFFICULTY_LEVELS)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Add threshold lines for bimodality indicators
    if ind_name == "bimodality_coefficient":
        ax.axhline(y=BIMODALITY_BC_THRESHOLD, color='gray', linestyle='--', alpha=0.5, label='BC threshold')
    elif ind_name == "silhouette_score_k2":
        ax.axhline(y=BIMODALITY_SIL_THRESHOLD, color='gray', linestyle='--', alpha=0.5, label='Sil threshold')

plt.tight_layout()
plt.show()

# --- Summary of success criteria from full experiment ---
success = metadata["success_criteria_assessment"]
print("\n=== Full Experiment Success Criteria ===")
print(f"  Flickering detected:        {success['flickering_detected']}")
print(f"  Models with bimodality:     {success['n_models_with_bimodality']}/3 ({', '.join(m.split('/')[-1] for m in success['models_with_bimodality'])})")
print(f"  Leading indicator found:    {success['leading_indicator_found']}")
print(f"  Variance scaling in range:  {success['variance_scaling_fit']}")
print(f"  Total experiment cost:      ${metadata['total_cost_usd']:.3f}")